# Graduate Probability — Theory Meets Simulation

This notebook teaches **core graduate-level probability** for students who know calculus and basic linear algebra. Each concept is stated precisely, then **verified with Python** so intuition and rigor reinforce each other.

## What you will learn

| Part | Topic | Why it matters |
|------|-------|----------------|
| 1 | **Probability spaces & axioms** | The language every proof uses |
| 2 | **Conditional probability & Bayes** | Updating beliefs with evidence |
| 3 | **Random variables (discrete)** | Counting outcomes, sums of trials |
| 4 | **Random variables (continuous)** | Densities, integrals, change of variables |
| 5 | **Expectation, variance, covariance** | The numbers practitioners actually compute |
| 6 | **Joint & conditional distributions** | Dependence between quantities |
| 7 | **Common distributions** | The vocabulary of applied probability |
| 8 | **Generating functions** | A compact handle on moments and sums |
| 9 | **Limit theorems** | Why the normal distribution appears everywhere |
| 10 | **Multivariate normal** | The backbone of statistics and finance |
| 11 | **Inequalities** | Bounding tails without full distributions |
| 12 | **Martingales** | Fair games, stopping times, and ruin probabilities |
| — | **Problem sets (5)** | Practice with hidden solutions |

## How to use this notebook

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — graduate probability is about *relationships*, not memorized formulas.
3. Sections end with **"Your turn"** exercises. The **problem sets** at the end have **click-to-reveal solutions** — try each problem before opening the answer.
4. Notation: \(P(A)\) is probability of event \(A\); \(E[X]\) is expected value of random variable \(X\); \(X \sim \mathcal{N}(\mu, \sigma^2)\) means \(X\) is normally distributed.

Let's begin.

---
## Setup — run this first

We use NumPy for arrays and simulation, SciPy for exact distributions, and Matplotlib for plots.

In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import comb

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", stats.__version__)

---
# Part 1 — Probability Spaces and Axioms

A **probability space** is a triple \((\Omega, \mathcal{F}, P)\):

- **Sample space** \(\Omega\): all possible outcomes of an experiment.
- **\(\sigma\)-algebra** \(\mathcal{F}\): the collection of events (subsets of \(\Omega\)) we can assign probability to.
- **Probability measure** \(P\): a function \(P: \mathcal{F} \to [0,1]\) satisfying the **Kolmogorov axioms**:
  1. \(P(\Omega) = 1\)
  2. \(P(A) \geq 0\) for all \(A \in \mathcal{F}\)
  3. **Countable additivity**: if \(A_1, A_2, \ldots\) are disjoint, then \(P\left(\bigcup_i A_i\right) = \sum_i P(A_i)\)

**Example:** Roll a fair six-sided die. \(\Omega = \{1,2,3,4,5,6\}\), \(P(\{k\}) = 1/6\) for each outcome \(k\).

For finite spaces, probabilities of events are sums of point masses. The code below estimates \(P(\text{sum} = 7)\) when rolling two fair dice by **Monte Carlo** — a technique we will use throughout.

In [ ]:
def monte_carlo_probability(event_fn, n_trials=100_000, seed=0):
    """Estimate P(event) by repeating an experiment and counting successes."""
    rng = np.random.default_rng(seed)
    outcomes = event_fn(rng, n_trials)
    return outcomes.mean()

def two_dice_sum_is_7(rng, n):
    die1 = rng.integers(1, 7, size=n)
    die2 = rng.integers(1, 7, size=n)
    return (die1 + die2) == 7

exact = 6 / 36  # six pairs (1,6),(2,5),... out of 36 outcomes
estimated = monte_carlo_probability(two_dice_sum_is_7, n_trials=200_000)

print(f"Exact P(sum=7)     = {exact:.6f}")
print(f"Monte Carlo estimate = {estimated:.6f}")

**Your turn:** Modify the event function to estimate \(P(\text{sum} \geq 10)\) for two dice. The exact answer is \(6/36\) as well — verify.

---
# Part 2 — Conditional Probability and Bayes' Rule

For events \(A, B\) with \(P(B) > 0\):

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

**Independence:** \(A\) and \(B\) are independent if \(P(A \cap B) = P(A)P(B)\), equivalently \(P(A \mid B) = P(A)\).

**Bayes' rule** updates a prior \(P(A)\) after observing \(B\):

$$P(A \mid B) = \frac{P(B \mid A)\, P(A)}{P(B)}$$

**Classic example (medical testing):** Disease prevalence \(P(D) = 0.01\), test sensitivity \(P(+ \mid D) = 0.99\), specificity \(P(- \mid \text{no } D) = 0.95\). What is \(P(D \mid +)\)?

Many students guess ~99%. Bayes shows the answer is much lower because the disease is rare.

In [ ]:
p_disease = 0.01
p_pos_given_disease = 0.99
p_neg_given_healthy = 0.95
p_pos_given_healthy = 1 - p_neg_given_healthy

# Law of total probability for P(+)
p_positive = (
    p_pos_given_disease * p_disease
    + p_pos_given_healthy * (1 - p_disease)
)

p_disease_given_positive = (p_pos_given_disease * p_disease) / p_positive

print(f"P(D | +) = {p_disease_given_positive:.4f}  ({p_disease_given_positive*100:.1f}%)")
print("Despite a 'good' test, most positives are false when the disease is rare.")

We can **simulate** the screening population and confirm Bayes numerically.

In [ ]:
n = 1_000_000
rng = np.random.default_rng(0)

has_disease = rng.random(n) < p_disease
test_positive = np.where(
    has_disease,
    rng.random(n) < p_pos_given_disease,
    rng.random(n) < p_pos_given_healthy,
)

simulated = has_disease[test_positive].mean()
print(f"Simulated P(D | +) = {simulated:.4f}")
print(f"Bayes formula      = {p_disease_given_positive:.4f}")

---
# Part 3 — Discrete Random Variables

A **random variable** \(X\) is a measurable function \(X: \Omega \to \mathbb{R}\). For discrete \(X\), the **probability mass function (PMF)** is \(p_X(x) = P(X = x)\).

Properties: \(p_X(x) \geq 0\) and \(\sum_x p_X(x) = 1\).

**Bernoulli** \(X \sim \text{Bern}(p)\): one trial, \(P(X=1)=p\), \(P(X=0)=1-p\).

**Binomial** \(X \sim \text{Bin}(n, p)\): number of successes in \(n\) independent Bernoulli trials:

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}, \quad k = 0, 1, \ldots, n$$

In [ ]:
n, p = 20, 0.3
k = np.arange(0, n + 1)

# Exact PMF via SciPy
binom_pmf = stats.binom.pmf(k, n, p)

# Manual PMF for comparison
manual_pmf = comb(n, k, exact=True) * p**k * (1 - p)**(n - k)
manual_pmf = manual_pmf / manual_pmf.sum()  # comb returns integers; normalize check

fig, ax = plt.subplots()
ax.bar(k, binom_pmf, alpha=0.7, label="SciPy binom.pmf")
ax.set_xlabel("k (number of successes)")
ax.set_ylabel("P(X = k)")
ax.set_title(f"Binomial(n={n}, p={p})")
ax.legend()
plt.show()

print(f"Sum of PMF (should be 1): {binom_pmf.sum():.6f}")
print(f"E[X] = np      = {n * p}")
print(f"E[X] = scipy   = {stats.binom.mean(n, p)}")

**Poisson** \(X \sim \text{Poisson}(\lambda)\): models counts in fixed intervals (arrivals, defaults per month):

$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}, \quad k = 0, 1, 2, \ldots$$

Poisson is the **limit of Binomial** when \(n \to \infty\), \(p \to 0\), \(np \to \lambda\).

In [ ]:
lambda_param = 4.0
n_large, p_small = 10_000, lambda_param / 10_000

k_plot = np.arange(0, 15)
poisson_pmf = stats.poisson.pmf(k_plot, lambda_param)
binom_approx = stats.binom.pmf(k_plot, n_large, p_small)

fig, ax = plt.subplots()
ax.plot(k_plot, poisson_pmf, "o-", label=f"Poisson(λ={lambda_param})")
ax.plot(k_plot, binom_approx, "s--", label=f"Binomial({n_large}, {p_small:.5f})")
ax.set_xlabel("k")
ax.set_ylabel("PMF")
ax.legend()
ax.set_title("Poisson as limit of Binomial")
plt.show()

---
# Part 4 — Continuous Random Variables

For continuous \(X\), probabilities are integrals of a **probability density function (PDF)** \(f_X\):

$$P(a \leq X \leq b) = \int_a^b f_X(x)\, dx, \quad f_X(x) \geq 0, \quad \int_{-\infty}^{\infty} f_X(x)\, dx = 1$$

The **cumulative distribution function (CDF)** is \(F_X(x) = P(X \leq x) = \int_{-\infty}^x f_X(t)\, dt\).

**Uniform** on \([a,b]\): \(f(x) = 1/(b-a)\).

**Exponential** \(X \sim \text{Exp}(\lambda)\): waiting times, \(f(x) = \lambda e^{-\lambda x}\) for \(x \geq 0\), memoryless property \(P(X > s+t \mid X > s) = P(X > t)\).

In [ ]:
lam = 2.0  # rate λ (mean = 1/λ)
x = np.linspace(0, 5, 500)

pdf = stats.expon.pdf(x, scale=1/lam)  # scipy uses scale = 1/λ
cdf = stats.expon.cdf(x, scale=1/lam)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, pdf)
axes[0].set_title(f"Exponential PDF (λ={lam})")
axes[0].set_xlabel("x")
axes[1].plot(x, cdf)
axes[1].set_title("CDF")
axes[1].set_xlabel("x")
plt.tight_layout()
plt.show()

# Memoryless property via simulation
rng = np.random.default_rng(1)
samples = rng.exponential(scale=1/lam, size=100_000)
survived = samples > 1.0
cond_mean_extra = (samples[survived] - 1.0).mean()
theoretical_mean = 1 / lam
print(f"E[X - 1 | X > 1] (simulated) = {cond_mean_extra:.4f}")
print(f"E[X] (theory, memoryless)    = {theoretical_mean:.4f}")

### Change of variables

If \(Y = g(X)\) with \(g\) strictly monotone, then

$$f_Y(y) = f_X(g^{-1}(y)) \left| \frac{d}{dy} g^{-1}(y) \right|$$

**Example:** \(X \sim \text{Uniform}(0,1)\), \(Y = -\ln X\). Then \(Y \sim \text{Exp}(1)\).

In [ ]:
rng = np.random.default_rng(2)
u = rng.uniform(0, 1, size=100_000)
y = -np.log(u)

fig, ax = plt.subplots()
ax.hist(y, bins=50, density=True, alpha=0.6, label="Simulated Y = -log(U)")
x_plot = np.linspace(0, 6, 200)
ax.plot(x_plot, stats.expon.pdf(x_plot), "r-", lw=2, label="Exp(1) PDF")
ax.legend()
ax.set_title("Change of variables: Uniform → Exponential")
plt.show()

---
# Part 5 — Expectation, Variance, and Covariance

**Expectation** (discrete): \(E[X] = \sum_x x\, p_X(x)\). (Continuous): \(E[X] = \int x f_X(x)\, dx\).

**Variance:** \(\text{Var}(X) = E[(X - E[X])^2] = E[X^2] - (E[X])^2\).

**Covariance:** \(\text{Cov}(X,Y) = E[(X - E[X])(Y - E[Y])] = E[XY] - E[X]E[Y]\).

**Linearity** (always): \(E[aX + bY] = aE[X] + bE[Y]\).

**Independence** implies \(\text{Cov}(X,Y) = 0\), but zero covariance does **not** imply independence (except for jointly normal — see Part 10).

In [ ]:
def sample_moments(x, y=None):
    """Return mean, variance, and optional covariance from samples."""
    mean_x = x.mean()
    var_x = x.var(ddof=0)  # population variance (divide by n)
    if y is None:
        return mean_x, var_x
    cov_xy = np.mean((x - mean_x) * (y - y.mean()))
    return mean_x, var_x, cov_xy

# X ~ Bernoulli(0.4)
rng = np.random.default_rng(3)
x = rng.binomial(1, 0.4, size=200_000)
mean_sim, var_sim = sample_moments(x)

p = 0.4
print("Bernoulli(0.4):")
print(f"  E[X] theory = {p:.4f}, simulated = {mean_sim:.4f}")
print(f"  Var(X) theory = {p*(1-p):.4f}, simulated = {var_sim:.4f}")

**Correlation** \(\rho_{X,Y} = \text{Cov}(X,Y) / (\sigma_X \sigma_Y)\) lies in \([-1, 1]\).

Below: \(X\) uniform on \([-1,1]\), \(Y = X^2\). They are **dependent** but \(\text{Cov}(X, Y) = 0\) (uncorrelated ≠ independent).

In [ ]:
rng = np.random.default_rng(4)
x = rng.uniform(-1, 1, size=50_000)
y = x**2

_, _, cov_xy = sample_moments(x, y)
corr = np.corrcoef(x, y)[0, 1]

fig, ax = plt.subplots()
ax.scatter(x[:2000], y[:2000], alpha=0.3, s=8)
ax.set_xlabel("X")
ax.set_ylabel("Y = X²")
ax.set_title(f"Dependent but uncorrelated: Cov = {cov_xy:.4f}, ρ = {corr:.4f}")
plt.show()

---
# Part 6 — Joint, Marginal, and Conditional Distributions

The **joint PDF** \(f_{X,Y}(x,y)\) satisfies \(P((X,Y) \in B) = \int_B f_{X,Y}(x,y)\, dx\, dy\).

- **Marginal:** \(f_X(x) = \int f_{X,Y}(x,y)\, dy\)
- **Conditional:** \(f_{Y|X}(y|x) = f_{X,Y}(x,y) / f_X(x)\) when \(f_X(x) > 0\)

**Bivariate normal** preview: if \((X,Y)\) are jointly normal, the conditional \(Y \mid X = x\) is also normal.

In [ ]:
# Joint: (X, Y) with Y = X + noise, X ~ N(0,1), noise ~ N(0, 0.5)
rng = np.random.default_rng(5)
n = 30_000
x = rng.normal(0, 1, size=n)
noise = rng.normal(0, 0.5, size=n)
y = x + noise

fig, ax = plt.subplots()
hb = ax.hexbin(x, y, gridsize=40, cmap="Blues", mincnt=1)
plt.colorbar(hb, ax=ax, label="count")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("Joint distribution: Y = X + ε")
plt.show()

# Conditional mean E[Y | X = x] should be x (linear regression line)
bins = np.linspace(-3, 3, 25)
bin_idx = np.digitize(x, bins)
cond_means = [y[bin_idx == i].mean() for i in range(1, len(bins))]
bin_centers = (bins[:-1] + bins[1:]) / 2

fig, ax = plt.subplots()
ax.scatter(bin_centers, cond_means, label="E[Y|X] (binned)")
ax.plot([-3, 3], [-3, 3], "r--", label="y = x")
ax.legend()
ax.set_xlabel("x")
ax.set_ylabel("E[Y | X ≈ x]")
plt.show()

---
# Part 7 — Common Distributions (Reference + Simulation)

Graduate courses assume fluency with this table. We plot PDFs and match sample means to theory.

In [ ]:
distributions = [
    ("Normal", stats.norm(0, 1)),
    ("Gamma", stats.gamma(2.0, scale=1.5)),
    ("Beta", stats.beta(2, 5)),
    ("Lognormal", stats.lognorm(s=0.5, scale=np.exp(0))),
    ("Student-t", stats.t(3)),
]

fig, axes = plt.subplots(len(distributions), 1, figsize=(9, 12))
rng = np.random.default_rng(6)

for ax, (name, dist) in zip(axes, distributions):
    samples = dist.rvs(size=50_000, random_state=rng)
    x_lo, x_hi = dist.ppf(0.001), dist.ppf(0.999)
    xs = np.linspace(x_lo, x_hi, 300)
    ax.hist(samples, bins=60, density=True, alpha=0.5, label="samples")
    ax.plot(xs, dist.pdf(xs), "r-", lw=2, label="PDF")
    ax.set_title(
        f"{name}: E={dist.mean():.3f}, Var={dist.var():.3f} | "
        f"sim E={samples.mean():.3f}, Var={samples.var():.3f}"
    )
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
# Part 8 — Moment Generating Functions and Characteristic Functions

The **moment generating function (MGF)** is \(M_X(t) = E[e^{tX}]\) when it exists. Moments follow: \(E[X^k] = M_X^{(k)}(0)\).

The **characteristic function** \(\phi_X(t) = E[e^{itX}]\) always exists and determines the distribution uniquely.

Key property: if \(X_1, \ldots, X_n\) are independent, then \(M_{S}(t) = \prod_i M_{X_i}(t)\) for \(S = \sum_i X_i\).

For \(X \sim \mathcal{N}(0,1)\): \(M_X(t) = e^{t^2/2}\), \(\phi_X(t) = e^{-t^2/2}\).

In [ ]:
# Estimate MGF of standard normal via Monte Carlo at several t values
rng = np.random.default_rng(7)
z = rng.normal(0, 1, size=500_000)

t_values = np.linspace(-1.5, 1.5, 7)
mgf_sim = [np.mean(np.exp(t * z)) for t in t_values]
mgf_theory = np.exp(t_values**2 / 2)

print("MGF of N(0,1): M(t) = exp(t²/2)")
for t, sim, th in zip(t_values, mgf_sim, mgf_theory):
    print(f"  t={t:+.1f}: simulated={sim:.4f}, theory={th:.4f}")

# Sum of independent Poissons is Poisson
lam1, lam2 = 2.0, 3.5
p1 = rng.poisson(lam1, size=200_000)
p2 = rng.poisson(lam2, size=200_000)
sum_poisson = p1 + p2

print(f"\nSum Poisson({lam1}) + Poisson({lam2}) → Poisson({lam1+lam2})")
print(f"  Simulated mean = {sum_poisson.mean():.3f}, theory = {lam1+lam2:.3f}")

---
# Part 9 — Limit Theorems

### Weak Law of Large Numbers (WLLN)

If \(X_1, \ldots, X_n\) are i.i.d. with \(E[|X_1|] < \infty\), then the sample mean \(\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i\) satisfies

$$\bar{X}_n \xrightarrow{P} E[X_1] \quad \text{(convergence in probability)}$$

### Central Limit Theorem (CLT)

If \(E[X_1^2] < \infty\), then

$$\frac{\bar{X}_n - E[X_1]}{\sqrt{\text{Var}(X_1)/n}} \xrightarrow{d} \mathcal{N}(0,1)$$

So sums of many small independent contributions become approximately normal — the reason Gaussian noise is everywhere.

In [ ]:
def clt_demo(dist, n_samples=10_000, sample_sizes=(1, 2, 5, 30, 100), seed=8):
    """Show sampling distribution of standardized mean approaching N(0,1)."""
    rng = np.random.default_rng(seed)
    mu, sigma = dist.mean(), dist.std()

    fig, axes = plt.subplots(1, len(sample_sizes), figsize=(14, 3.5), sharey=True)
    for ax, n in zip(axes, sample_sizes):
        # n_samples rows, n columns → sample means
        data = dist.rvs(size=(n_samples, n), random_state=rng)
        means = data.mean(axis=1)
        standardized = (means - mu) / (sigma / np.sqrt(n))
        ax.hist(standardized, bins=40, density=True, alpha=0.6)
        xs = np.linspace(-4, 4, 200)
        ax.plot(xs, stats.norm.pdf(xs), "r-", lw=2)
        ax.set_title(f"n = {n}")
    fig.suptitle(f"CLT: standardized mean (underlying: {dist.name})")
    plt.tight_layout()
    plt.show()

# Uniform is not normal, but the mean of many uniforms is
clt_demo(stats.uniform(0, 1))

In [ ]:
# CLT even works for Bernoulli — binomial as sum of Bernoullis
clt_demo(stats.bernoulli(0.3), sample_sizes=(1, 5, 20, 50, 200))

**Your turn:** Run `clt_demo` on an exponential distribution. Does convergence look faster or slower than for the uniform?

---
# Part 10 — Multivariate Normal Distribution

A random vector \(X \in \mathbb{R}^d\) is **multivariate normal** \(X \sim \mathcal{N}(\mu, \Sigma)\) if every linear combination \(a^\top X\) is univariate normal.

PDF (when \(\Sigma\) is invertible):

$$f(x) = \frac{1}{(2\pi)^{d/2} |\Sigma|^{1/2}} \exp\left(-\frac{1}{2}(x-\mu)^\top \Sigma^{-1} (x-\mu)\right)$$

Key facts:
- Marginals are normal.
- \(X\) and \(Y\) are independent **iff** \(\Sigma_{12} = 0\) (for jointly normal).
- If \(X \sim \mathcal{N}(0, I_d)\), then \(AX\) is multivariate normal with covariance \(AA^\top\).

In [ ]:
rng = np.random.default_rng(9)

mu = np.array([1.0, -0.5])
Sigma = np.array([[1.0, 0.8], [0.8, 1.5]])

samples = rng.multivariate_normal(mu, Sigma, size=20_000)

fig, ax = plt.subplots()
ax.scatter(samples[:, 0], samples[:, 1], alpha=0.2, s=5)
ax.set_xlabel("X₁")
ax.set_ylabel("X₂")
ax.set_title("Bivariate normal samples (ρ ≈ 0.8)")
plt.show()

print("Sample covariance matrix:")
print(np.cov(samples, rowvar=False))
print("\nTheoretical Σ:")
print(Sigma)

### Linear algebra connection

Any covariance matrix \(\Sigma\) is positive semi-definite. We can **whiten** data: if \(Z \sim \mathcal{N}(0, I)\), then \(X = \mu + L Z\) where \(L L^\top = \Sigma\) (e.g. Cholesky factor).

In [ ]:
L = np.linalg.cholesky(Sigma)
z = rng.standard_normal(size=(20_000, 2))
x_from_cholesky = mu + z @ L.T

print("Covariance via Cholesky sampling:")
print(np.cov(x_from_cholesky, rowvar=False))

---
# Part 11 — Inequalities You Will Use in Proofs

These bound tail probabilities without knowing the full distribution.

**Markov:** If \(X \geq 0\), then \(P(X \geq a) \leq E[X]/a\) for \(a > 0\).

**Chebyshev:** \(P(|X - E[X]| \geq k\sigma) \leq 1/k^2\).

**Jensen:** For convex \(\varphi\), \(E[\varphi(X)] \geq \varphi(E[X])\). Example: \(E[X^2] \geq (E[X])^2\).

Below we verify Chebyshev on exponential data.

In [ ]:
lam = 1.0
dist = stats.expon(scale=1/lam)
mu, sigma = dist.mean(), dist.std()

rng = np.random.default_rng(10)
x = dist.rvs(size=500_000, random_state=rng)

k_values = np.array([1, 2, 3, 4, 5])
tail_prob = [np.mean(np.abs(x - mu) >= k * sigma) for k in k_values]
chebyshev_bound = 1 / k_values**2

print("k | P(|X-μ| ≥ kσ) | Chebyshev ≤ 1/k²")
for k, p, b in zip(k_values, tail_prob, chebyshev_bound):
    print(f"{k} | {p:.4f}         | {b:.4f}")

---
# Part 12 — Martingales

A **filtration** \(\{\mathcal{F}_n\}_{n \geq 0}\) is an increasing family of \(\sigma\)-algebras: \(\mathcal{F}_0 \subseteq \mathcal{F}_1 \subseteq \cdots\). Think of \(\mathcal{F}_n\) as *everything observable by time \(n\)*.

A sequence \(\{M_n\}\) is **adapted** if \(M_n\) is measurable with respect to \(\mathcal{F}_n\) for each \(n\).

**Martingale:** An adapted sequence with \(E[|M_n|] < \infty\) satisfying

$$E[M_{n+1} \mid \mathcal{F}_n] = M_n \quad \text{a.s.}$$

**Fair-game interpretation:** Given all history up to now, your expected wealth tomorrow equals today's wealth — no betting strategy can beat the house *in expectation* if the game is martingale.

**Submartingale:** \(E[M_{n+1} \mid \mathcal{F}_n] \geq M_n\) (tends to rise). **Supermartingale:** \(E[M_{n+1} \mid \mathcal{F}_n] \leq M_n\) (tends to fall).

### Canonical examples

1. **Simple symmetric random walk:** \(S_0 = 0\), \(S_{n+1} = S_n + X_{n+1}\) with \(P(X_i = \pm 1) = 1/2\). Then \(S_n\) is a martingale.

2. **Quadratic variation correction:** For the same walk, \(M_n = S_n^2 - n\) is a martingale because one step changes \(S^2\) by \(\pm 2S_n \pm 1\) with equal probability, so \(E[S_{n+1}^2 \mid S_n] = S_n^2 + 1\).

3. **Doob transform:** If \(M_n\) is a martingale and \(H_n\) is a bounded predictable strategy (bet \(H_n\) at time \(n-1\)), then \(\sum_{k=1}^n H_k (M_k - M_{k-1})\) is a martingale.

In [ ]:
# Verify S_n is a martingale: E[S_{n+1} | S_n] = S_n for symmetric random walk
rng = np.random.default_rng(12)
n_steps = 200
n_paths = 50_000

# Build random walks: each path is cumulative sum of ±1 steps
steps = rng.choice([-1, 1], size=(n_paths, n_steps))
s_n = np.cumsum(steps, axis=1)  # shape (paths, n_steps)

# At each time n, check E[S_{n+1} | S_n = s] ≈ s across bins of S_n
n_check = 80
s_current = s_n[:, n_check - 1]
s_next = s_n[:, n_check]

bins = np.arange(-20, 21, 2)
bin_idx = np.digitize(s_current, bins)

fig, ax = plt.subplots()
for i in range(1, len(bins)):
    mask = bin_idx == i
    if mask.sum() < 200:
        continue
    s_bin = s_current[mask].mean()
    next_mean = s_next[mask].mean()
    ax.scatter(s_bin, next_mean, s=40)

ax.plot([-20, 20], [-20, 20], "r--", label="y = x (martingale)")
ax.set_xlabel("S_n (binned mean)")
ax.set_ylabel("E[S_{n+1} | S_n ≈ s]")
ax.set_title("Symmetric random walk: conditional expectation ≈ identity")
ax.legend()
plt.show()

# Verify S_n^2 - n is a martingale
m_current = s_current**2 - (n_check - 1)
m_next = s_next**2 - n_check
print(f"E[M_n] at n={n_check-1}: {m_current.mean():.4f} (theory: 0)")
print(f"E[M_{n_check}] at n={n_check}: {m_next.mean():.4f} (theory: 0)")
print(f"E[M_{n_check} - M_{n_check-1} | paths]: {np.mean(m_next - m_current):.6f} (theory: 0)")

### Optional stopping and gambler's ruin

A **stopping time** \(\tau\) is a random time such that \(\{\tau = n\} \in \mathcal{F}_n\) for each \(n\) — you decide to stop using only information available so far (no peeking into the future).

**Doob's optional stopping theorem (OST):** Under integrability and bounded-stop conditions, \(E[M_\tau] = E[M_0]\) for a martingale \(M_n\).

**Gambler's ruin:** Start with \(k\) dollars, bet \$1 each round on a fair coin. Stop when wealth hits \(0\) or \(N\). Let \(\tau\) be the stopping time. Since \(S_n\) is a martingale and \(S_0 = k\),

$$E[S_\tau] = k = 0 \cdot P(\text{ruin}) + N \cdot P(\text{reach } N)$$

So \(P(\text{reach } N) = k/N\) and \(P(\text{ruin}) = 1 - k/N\).

**Martingale convergence:** A nonnegative supermartingale converges almost surely (Doob). Bounded martingales converge a.s. and in \(L^1\).

In [ ]:
def gambler_ruin_sim(k, N, n_sims=50_000, max_rounds=10_000, seed=13):
    """Simulate fair gambler's ruin; return fraction reaching N before 0."""
    rng = np.random.default_rng(seed)
    reached = np.zeros(n_sims, dtype=bool)

    for i in range(n_sims):
        wealth = k
        for _ in range(max_rounds):
            if wealth <= 0 or wealth >= N:
                break
            wealth += rng.choice([-1, 1])
        reached[i] = wealth >= N

    return reached.mean()

k, N = 25, 100
sim_prob = gambler_ruin_sim(k, N)
theory_prob = k / N

print(f"Gambler's ruin: start={k}, barrier={N}")
print(f"  Simulated P(reach {N}) = {sim_prob:.4f}")
print(f"  Theory k/N           = {theory_prob:.4f}")

# Plot sample paths
rng = np.random.default_rng(14)
fig, ax = plt.subplots()
for _ in range(30):
    wealth = k
    path = [wealth]
    for _ in range(500):
        if wealth <= 0 or wealth >= N:
            break
        wealth += rng.choice([-1, 1])
        path.append(wealth)
    ax.plot(path, alpha=0.6)
ax.axhline(0, color="k", lw=1)
ax.axhline(N, color="k", lw=1)
ax.set_xlabel("round")
ax.set_ylabel("wealth")
ax.set_title(f"Sample paths: gambler's ruin (k={k}, N={N})")
plt.show()

---
# Problem Sets

Five problem sets cover the full notebook. **Do not open solutions until you have tried each problem.**

Solutions are hidden below each set — click **"Reveal solution"** to expand.

| Set | Topics |
|-----|--------|
| 1 | Axioms, conditioning, Bayes |
| 2 | Discrete & continuous RVs |
| 3 | Joint distributions, expectation |
| 4 | MGFs, inequalities, limit theorems |
| 5 | Martingales & multivariate normal |

## Problem Set 1 — Foundations

1. Using only the Kolmogorov axioms, prove that \(P(A^c) = 1 - P(A)\) for any event \(A\).

2. Two fair dice are rolled. Find \(P(\text{sum} \geq 10)\) exactly by counting outcomes.

3. A disease has prevalence \(2\%\). A test has sensitivity \(95\%\) and specificity \(90\%\). Compute \(P(\text{disease} \mid +)\).

4. Events \(A\) and \(B\) satisfy \(P(A) = 0.4\), \(P(B) = 0.5\), \(P(A \cup B) = 0.7\). Are \(A\) and \(B\) independent? Find \(P(A \mid B)\).

<details>
<summary><strong>Reveal solution — Problem Set 1</strong></summary>

**1.** \(\Omega = A \cup A^c\) disjoint union, so \(P(\Omega) = P(A) + P(A^c)\). Axiom 1 gives \(1 = P(A) + P(A^c)\).

**2.** Pairs with sum ≥ 10: (4,6), (5,5), (6,4), (5,6), (6,5), (6,6) → **6 outcomes out of 36**, so \(P = 1/6\).

**3.** Bayes: \(P(D \mid +) = \frac{0.95 \times 0.02}{0.95 \times 0.02 + 0.10 \times 0.98} = \frac{0.019}{0.019 + 0.098} = \frac{0.019}{0.117} \approx \mathbf{0.162}\) (16.2%).

**4.** \(P(A \cap B) = P(A) + P(B) - P(A \cup B) = 0.2\). Independence would require \(0.2 = 0.4 \times 0.5 = 0.2\) — **yes, independent**. \(P(A \mid B) = P(A \cap B)/P(B) = 0.2/0.5 = \mathbf{0.4}\).

</details>

## Problem Set 2 — Random Variables

1. For \(X \sim \text{Bin}(n,p)\), derive \(\text{Var}(X) = np(1-p)\) using \(E[X^2] = E[X(X-1)] + E[X]\) and the MGF or indicator trick.

2. Prove the memoryless property of the exponential: \(P(X > s+t \mid X > s) = P(X > t)\) for \(s,t \geq 0\).

3. Let \(X \sim \text{Uniform}(0,1)\) and \(Y = X^2\). Find the PDF of \(Y\) on \((0,1)\).

4. For \(X \sim \text{Bin}(200, 0.02)\), approximate \(P(X \geq 6)\) using a Poisson approximation. Compare to the exact binomial value.

<details>
<summary><strong>Reveal solution — Problem Set 2</strong></summary>

**1.** For \(k \geq 2\): \(k(k-1)\binom{n}{k} = n(n-1)\binom{n-2}{k-2}\). Sum over \(k\) gives \(E[X(X-1)] = n(n-1)p^2\), so \(E[X^2] = n(n-1)p^2 + np\). Then \(\text{Var}(X) = np - n^2p^2 = np(1-p)\).

**2.** \(P(X > s+t \mid X > s) = \frac{P(X > s+t)}{P(X > s)} = \frac{e^{-\lambda(s+t)}}{e^{-\lambda s}} = e^{-\lambda t} = P(X > t)\).

**3.** For \(0 < y < 1\): \(F_Y(y) = P(X^2 \leq y) = P(X \leq \sqrt{y}) = \sqrt{y}\). So \(f_Y(y) = \frac{1}{2\sqrt{y}}\) on \((0,1)\).

**4.** Poisson with \(\lambda = np = 4\): \(P(X \geq 6) = 1 - \sum_{k=0}^{5} e^{-4}4^k/k! \approx 0.2149\). Exact binomial: **≈ 0.2179** (close agreement).

</details>

## Problem Set 3 — Joint Distributions & Expectation

1. Prove \(\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y) + 2\,\text{Cov}(X,Y)\).

2. Joint PMF: \(P(X=0,Y=0)=0.1\), \(P(0,1)=0.2\), \(P(1,0)=0.3\), \(P(1,1)=0.4\). Find \(E[XY]\) and \(\text{Cov}(X,Y)\).

3. Let \(X \sim \text{Uniform}(0,1)\) and \(Y \mid X = x \sim \text{Uniform}(0, x)\). Find \(E[Y]\) and \(E[X \mid Y]\) (qualitative: is it linear in \(Y\)?).

4. State and justify the **tower property**: \(E[E[X \mid Y]] = E[X]\).

<details>
<summary><strong>Reveal solution — Problem Set 3</strong></summary>

**1.** Expand \(\text{Var}(X+Y) = E[(X+Y)^2] - (E[X]+E[Y])^2\), expand squares, and collect cross-term \(2E[XY] - 2E[X]E[Y] = 2\,\text{Cov}(X,Y)\).

**2.** \(E[X]=0.7\), \(E[Y]=0.6\), \(E[XY] = 0 \cdot 0 \cdot 0.1 + \cdots + 1\cdot 1\cdot 0.4 = 0.4\). \(\text{Cov} = 0.4 - 0.42 = \mathbf{-0.02}\).

**3.** \(E[Y \mid X=x] = x/2\), so by tower \(E[Y] = E[X/2] = 1/4\). For \(E[X \mid Y]\): given \(Y=y\), \(X\) is uniform on \([y,1]\), so \(E[X \mid Y=y] = (1+y)/2\) — **linear in \(Y\)** with slope \(1/2\).

**4.** Tower property: averaging the conditional expectation over \(Y\) recovers the unconditional mean. Proof: \(E[E[X|Y]] = \sum_y E[X|Y=y] P(Y=y) = \sum_y \sum_x x P(X=x|Y=y)P(Y=y) = \sum_x x P(X=x) = E[X]\).

</details>

## Problem Set 4 — MGFs, Inequalities & Limit Theorems

1. Find the MGF of \(X \sim \text{Exp}(\lambda)\) and use it to compute \(E[X^2]\).

2. A factory produces items with mean weight 100g and variance 25g². Using Chebyshev, bound \(P(|X - 100| \geq 15)\).

3. A fair coin is flipped 10,000 times. Approximate \(P(4900 \leq S \leq 5100)\) where \(S\) is the number of heads, using the CLT.

4. Explain in words why the CLT does **not** apply to the maximum of i.i.d. exponentials as \(n \to \infty\) (what distribution does the max approach instead?).

<details>
<summary><strong>Reveal solution — Problem Set 4</strong></summary>

**1.** \(M_X(t) = E[e^{tX}] = \int_0^\infty e^{tx}\lambda e^{-\lambda x}dx = \frac{\lambda}{\lambda - t}\) for \(t < \lambda\). \(M_X'(t) = \frac{\lambda}{(\lambda-t)^2}\), \(M_X''(t) = \frac{2\lambda}{(\lambda-t)^3}\). At \(t=0\): \(E[X] = 1/\lambda\), \(E[X^2] = 2/\lambda^2\).

**2.** \(\sigma = 5\). Chebyshev: \(P(|X-100| \geq 15) = P(|X-100| \geq 3\sigma) \leq 1/9 \approx \mathbf{0.111}\).

**3.** \(S \approx \mathcal{N}(5000, 2500)\). Standardize: \(P(4900 \leq S \leq 5100) \approx P(-2 \leq Z \leq 2) \approx 0.9545\).

**4.** The max grows to the right tail; normalized max converges to **Gumbel / extreme-value** distribution, not Gaussian. CLT applies to sums/averages of fixed-mean contributions, not to order statistics driven by tail behavior.

</details>

## Problem Set 5 — Martingales & Multivariate Normal

1. Let \(M_n\) be a martingale with \(M_0 = 0\). Show that \(E[M_n] = 0\) for all \(n\).

2. For symmetric random walk \(S_n\), verify that \(M_n = S_n^2 - n\) is a martingale by computing \(E[S_{n+1}^2 \mid S_n]\).

3. Gambler's ruin: start with \$30, stop at \$0 or \$100, fair game. Find ruin probability and expected final wealth using optional stopping.

4. For \((X_1, X_2) \sim \mathcal{N}(0, \Sigma)\) with \(\Sigma = \begin{pmatrix}1 & \rho \\ \rho & 1\end{pmatrix}\), derive \(E[X_2 \mid X_1 = x]\) and \(\text{Var}(X_2 \mid X_1 = x)\).

<details>
<summary><strong>Reveal solution — Problem Set 5</strong></summary>

**1.** By induction: \(E[M_1] = E[E[M_1 \mid \mathcal{F}_0]] = E[M_0] = 0\). If \(E[M_n]=0\), then \(E[M_{n+1}] = E[E[M_{n+1} \mid \mathcal{F}_n]] = E[M_n] = 0\).

**2.** \(S_{n+1} = S_n + D\) with \(D \in \{-1,1\}\) fair. \(S_{n+1}^2 = S_n^2 + 2S_n D + D^2\). Given \(S_n\), \(E[2S_n D] = 2S_n E[D] = 0\) and \(E[D^2]=1\). So \(E[S_{n+1}^2 \mid S_n] = S_n^2 + 1\), hence \(E[M_{n+1} \mid S_n] = S_n^2 + 1 - (n+1) = S_n^2 - n = M_n\).

**3.** Martingale \(S_n\), \(S_0=30\), barriers 0 and 100. OST: \(E[S_\tau] = 30 = 0 \cdot P(\text{ruin}) + 100 \cdot P(\text{win})\). So \(P(\text{win}) = 0.3\), **\(P(\text{ruin}) = 0.7\)**. Expected final wealth is \(30\) (not the endpoint value — OST gives the *martingale* expectation).

**4.** Conditional normal formulas: \(E[X_2 \mid X_1=x] = \rho x\), \(\text{Var}(X_2 \mid X_1=x) = 1 - \rho^2\).

</details>

---
### Optional verification code (run after revealing solutions)

The cells below numerically check selected problems. **Spoiler warning** — only run after you have solved the problem on paper.

In [ ]:
# PS1 #3 — Bayes verification
p_d, sens, spec = 0.02, 0.95, 0.90
p_pos = sens * p_d + (1 - spec) * (1 - p_d)
print(f"PS1 #3: P(D|+) = {sens * p_d / p_pos:.4f}")

# PS2 #4 — Poisson vs binomial tail
exact = 1 - stats.binom.cdf(5, 200, 0.02)
poisson = 1 - stats.poisson.cdf(5, 4)
print(f"PS2 #4: exact binomial P(X>=6) = {exact:.4f}, Poisson approx = {poisson:.4f}")

In [ ]:
# PS3 #2 — covariance from joint PMF table
E_XY = 0.4
E_X, E_Y = 0.7, 0.6
print(f"PS3 #2: Cov(X,Y) = {E_XY - E_X * E_Y:.4f}")

# PS4 #3 — CLT coin flip approximation
n_flips = 10_000
mu, sigma = n_flips * 0.5, np.sqrt(n_flips * 0.25)
clt_approx = stats.norm.cdf(5100, mu, sigma) - stats.norm.cdf(4900, mu, sigma)
print(f"PS4 #3: CLT approx P(4900<=S<=5100) = {clt_approx:.4f}")

# PS5 #4 — conditional normal regression
rho = 0.75
rng = np.random.default_rng(15)
samples = rng.multivariate_normal([0, 0], [[1, rho], [rho, 1]], size=50_000)
slope = np.polyfit(samples[:, 0], samples[:, 1], 1)[0]
print(f"PS5 #4: regression slope ≈ ρ: {slope:.4f} (ρ={rho})")

---
## Further reading

- **Durrett**, *Probability: Theory and Examples* — rigorous graduate text with measure theory and martingales.
- **Blitzstein & Hwang**, *Introduction to Probability* — excellent intuition and problems.
- **Williams**, *Probability with Martingales* — martingale-centric graduate introduction.
- **Casella & Berger**, *Statistical Inference* — bridges probability to statistics.

You now have the computational companion to those proofs: whenever a theorem says "converges in distribution," you can **simulate it** and see the histogram approach the limit.